In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch:", torch.__version__)

class GeometryCNN(nn.Module):
    def __init__(self):
        super(GeometryCNN, self).__init__()
        # Layer 1: Input 64x64x3 -> Output 62x62x16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3)
        # Layer 2: Max Pool -> Output 31x31x16
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Layer 3: Conv -> Output 29x29x32
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        # Layer 4: Max Pool -> Output 14x14x32
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Layer 5: Conv -> Output 12x12x64
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)
        # Fully Connected
        self.fc = nn.Linear(64 * 12 * 12, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        print(f"after conv1: {x.shape}")
        x = self.pool1(x)
        print(f"after pool1: {x.shape}")
        x = F.relu(self.conv2(x))
        print(f"after conv2: {x.shape}")
        x = self.pool2(x)
        print(f"after pool2: {x.shape}")
        x = F.relu(self.conv3(x))
        print(f"after conv3: {x.shape}")
        x = x.view(x.size(0), -1)
        print(f"after flatten: {x.shape}")
        x = self.fc(x)
        print(f"output: {x.shape}")
        return x


model = GeometryCNN()
dummy_input = torch.randn(1, 3, 64, 64)

print("=== verifying shapes ===")
with torch.no_grad():
    model(dummy_input)

# param count
total = sum(p.numel() for p in model.parameters())
print(f"total params: {total:,}\n")

for name, p in model.named_parameters():
    print(f"{name}: {p.numel()} params, shape = {list(p.shape)}")

# parameter explosion - without pooling
class GeometryCNN_NoPool(nn.Module):
    def __init__(self):
        super(GeometryCNN_NoPool, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3)   # 64->62
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)  # 62->60
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)  # 60->58
        self.fc    = nn.Linear(64 * 58 * 58, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)


model_pool   = GeometryCNN()
model_nopool = GeometryCNN_NoPool()

p_pool   = sum(p.numel() for p in model_pool.parameters())
p_nopool = sum(p.numel() for p in model_nopool.parameters())

print(f"with pooling   : {p_pool:,} params")
print(f"without pooling: {p_nopool:,} params")
print(f"ratio: {p_nopool / p_pool:.1f}x more params without pooling")

fc_pool   = 64 * 12 * 12 * 10 + 10
fc_nopool = 64 * 58 * 58 * 10 + 10
print(f"\nFC layer only:")
print(f"  with pool:    {fc_pool:,}")
print(f"  without pool: {fc_nopool:,}")
print(f"  factor: {fc_nopool // fc_pool}x more")

torch: 2.10.0+cpu
=== verifying shapes ===
after conv1: torch.Size([1, 16, 62, 62])
after pool1: torch.Size([1, 16, 31, 31])
after conv2: torch.Size([1, 32, 29, 29])
after pool2: torch.Size([1, 32, 14, 14])
after conv3: torch.Size([1, 64, 12, 12])
after flatten: torch.Size([1, 9216])
output: torch.Size([1, 10])
total params: 115,754

conv1.weight: 432 params, shape = [16, 3, 3, 3]
conv1.bias: 16 params, shape = [16]
conv2.weight: 4608 params, shape = [32, 16, 3, 3]
conv2.bias: 32 params, shape = [32]
conv3.weight: 18432 params, shape = [64, 32, 3, 3]
conv3.bias: 64 params, shape = [64]
fc.weight: 92160 params, shape = [10, 9216]
fc.bias: 10 params, shape = [10]
with pooling   : 115,754 params
without pooling: 2,176,554 params
ratio: 18.8x more params without pooling

FC layer only:
  with pool:    92,170
  without pool: 2,152,970
  factor: 23x more
